In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/luisfelipevendramim/supchain/model_metrics.csv
/kaggle/input/datasets/luisfelipevendramim/supchain/inventory_results.csv
/kaggle/input/datasets/luisfelipevendramim/supchain/inventory_projection.csv
/kaggle/input/datasets/luisfelipevendramim/supchain/forecast_results.csv
/kaggle/input/datasets/luisfelipevendramim/supchain/inventory_kpis.csv
/kaggle/input/datasets/luisfelipevendramim/supchain/forecast_scenarios.csv


1. Load Data
2. Executive Overview
3. Demand Forecast
4. Inventory Projection
5. Scenario Analysis
6. Model Performance
7. Business Impact
8. Executive Summary

In [2]:
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display

**1 — Load Data**

In [3]:
forecast = pd.read_csv(
    "/kaggle/input/datasets/luisfelipevendramim/supchain/forecast_results.csv"
)

metrics = pd.read_csv(
    "/kaggle/input/datasets/luisfelipevendramim/supchain/model_metrics.csv"
)

inventory = pd.read_csv(
    "/kaggle/input/datasets/luisfelipevendramim/supchain/inventory_results.csv"
)

projection = pd.read_csv(
    "/kaggle/input/datasets/luisfelipevendramim/supchain/inventory_projection.csv"
)

scenarios = pd.read_csv(
    "/kaggle/input/datasets/luisfelipevendramim/supchain/forecast_scenarios.csv"
)

kpis = pd.read_csv(
    "/kaggle/input/datasets/luisfelipevendramim/supchain/inventory_kpis.csv"
)

**2 — Executive Overview**

In [4]:
print("="*50)
print("EXECUTIVE OVERVIEW")
print("="*50)

print(
    f"Mean Demand: {inventory['Mean_Demand'].iloc[0]:,.0f}"
)

print(
    f"Safety Stock: {inventory['Safety_Stock'].iloc[0]:,.0f}"
)

print(
    f"ROP: {inventory['ROP'].iloc[0]:,.0f}"
)

print(
    f"EOQ: {inventory['EOQ'].iloc[0]:,.0f}"
)

EXECUTIVE OVERVIEW
Mean Demand: 2,597
Safety Stock: 3,637
ROP: 21,819
EOQ: 6,158


**3 — Demand Forecast**

In [5]:
forecast["Date"] = pd.to_datetime(
    forecast["Date"]
)

In [6]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=forecast["Date"],
        y=forecast["Actual"],
        name="Actual"
    )
)

fig.add_trace(
    go.Scatter(
        x=forecast["Date"],
        y=forecast["XGBoost"],
        name="XGBoost"
    )
)

fig.add_trace(
    go.Scatter(
        x=forecast["Date"],
        y=forecast["LightGBM"],
        name="LightGBM"
    )
)

fig.add_trace(
    go.Scatter(
        x=forecast["Date"],
        y=forecast["Prophet"],
        name="Prophet"
    )
)

fig.update_layout(
    title="Demand Forecast Comparison",
    height=600
)

fig.show()

**4 — Inventory Projection**

In [7]:
fig_inventory = go.Figure()

fig_inventory.add_trace(
    go.Scatter(
        y=projection["inventory"],
        name="Inventory"
    )
)

fig_inventory.add_hline(
    y=inventory["ROP"].iloc[0],
    line_dash="dash"
)

fig_inventory.update_layout(
    title="Inventory Projection"
)

fig_inventory.show()

**5 — Scenario Analysis**

In [8]:
fig_scenario = go.Figure()

for col in [
    "optimistic",
    "base",
    "pessimistic"
]:

    fig_scenario.add_trace(
        go.Scatter(
            y=scenarios[col],
            name=col
        )
    )

fig_scenario.update_layout(
    title="Scenario Analysis"
)

fig_scenario.show()

**6 — Model Performance**

In [9]:
fig_perf = px.bar(
    metrics,
    x="Model",
    y="MAE",
    text="MAE"
)

fig_perf.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside'
)

fig_perf.show()

**7 — Business Impact**

In [10]:
naive_mae = metrics[
    metrics["Model"]=="Naive"
]["MAE"].iloc[0]

xgb_mae = metrics[
    metrics["Model"]=="XGBoost"
]["MAE"].iloc[0]

improvement = (
    (
        naive_mae - xgb_mae
    )
    /
    naive_mae
) * 100

print(
    f"Forecast Error Reduction: {improvement:.1f}%"
)

Forecast Error Reduction: 37.1%


**8 — Executive Summary**

In [11]:
best_model = (
    metrics
    .sort_values("MAE")
    .iloc[0]["Model"]
)

print("="*50)

print("EXECUTIVE SUMMARY")

print("="*50)

print(
    f"Best Model: {best_model}"
)

print(
    f"Forecast Error Reduction: {improvement:.1f}%"
)

print(
    f"Recommended EOQ: {inventory['EOQ'].iloc[0]:.0f}"
)

print(
    f"Recommended ROP: {inventory['ROP'].iloc[0]:.0f}"
)

EXECUTIVE SUMMARY
Best Model: XGBoost
Forecast Error Reduction: 37.1%
Recommended EOQ: 6158
Recommended ROP: 21819
